# Imports

In [12]:
import os
import argparse

import numpy as np
from datasets import load_dataset
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase
from deepeval.test_case import LLMTestCaseParams
from dotenv import load_dotenv
from openai import OpenAI
from scipy.stats import ttest_rel, ttest_1samp
import json

# OpenAI Token

In [2]:
os.environ["OPENAI_API_KEY"] = "A tokena nema("

# Working Part

In [3]:
# Выдает, ошибку, не знаю почему...
#!conda activate dl_course && python evaluation_openai.py --num_samples 30

In [6]:
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

client = OpenAI()

style_matching_metric_geval = GEval(
    name="Style Matching",
    criteria="Определи, насколько последняя фраза (actual output) делает анекдот смешным, логичным и остроумным:",

    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT]
)

humor_quality_metric = GEval(
    name="Humor Quality",
    criteria="Оцени, насколько последняя фраза (actual output) смешная. "
             "Учитывай качество шутки, её оригинальность, уместность и способность вызвать улыбку. "
             "Ответ должен быть логичным и доставлять удовольствие при чтении.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT]
)

system_message1 = {"role": "system",
                   "content": "Ты чат бот, который генерирует анекдоты. "
                              "Продолжи диалог одной репликой так, чтоб получился анекдот. "
                              "Example input: - Я не могу выйти на работу, потому что жена сломала два ребра. - А при чём тут ты? "
                              "Expected output: - Мои!"}
system_message2 = {"role": "system",
                   "content": "Ты чат бот, который генерирует диалоги. Продолжи диалог"}


def prepare_openai_jokes_dataset(args):
    ds = load_dataset("inkoziev/jokes_dialogues")
    ds_df = ds['train'].to_pandas()

    # print(ds_df)
    unique_chat_ids = ds_df['src_hash'].unique()
    # unique_chat_ids = set(ds['train']['src_hash'])
    sampled_chat_ids = np.random.choice(list(unique_chat_ids), args.num_samples, replace=False)

    openai_chat_examples = []
    for chat_id in sampled_chat_ids:
        reply_nums = ds_df[ds_df['src_hash'] == chat_id]['reply_num'].values
        reply_nums = np.sort(reply_nums)
        messages = [system_message1]
        if reply_nums[0] != 1:
            continue
        message1 = ds_df[(ds_df['src_hash'] == chat_id) & (ds_df['reply_num'] == 1)]['context'].values[0]
        messages.append({"role": "assistant" if 0 % 2 == reply_nums[-1] % 2 else "user",
                         "content": message1})

        for reply_num in reply_nums:
            message = ds_df[(ds_df['src_hash'] == chat_id) & (ds_df['reply_num'] == reply_num)]['utterance'].values[0]
            message_metadata = {"role": "assistant" if reply_num % 2 == reply_nums[-1] % 2 else "user",
                                "content": message}
            if message_metadata['role'] == 'assistant':
                if reply_num == reply_nums[-1]:
                    message_metadata['weight'] = 1
                else:
                    message_metadata['weight'] = 0
            messages.append(message_metadata)

        openai_chat_example = {"messages": messages}
        openai_chat_examples.append(openai_chat_example)
    return openai_chat_examples


def get_model_predictions(gt_examples, system_message):
    predicted_examples = []
    for i_example in range(len(gt_examples)):
        completion = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[system_message] + gt_examples[i_example]['messages'][1:-1],
        )
        # predicted_examples.append(completion.choices[0].message.content)
        predicted_example = [system_message] + gt_examples[i_example]['messages'][1:-1]
        predicted_example.append({"role": "assistant", "content": completion.choices[0].message.content})
        predicted_example = {"messages": predicted_example}
        predicted_examples.append(predicted_example)

    return predicted_examples


def evaluate_metrics(gt_examples, predicted_examples1, predicted_examples2):
    scores1 = []
    scores2 = []
    scores1_quality = []
    scores2_quality = []

    for i_example in range(len(gt_examples)):
        test_case1 = LLMTestCase(
            input="\n".join([msg['content'] for msg in predicted_examples1[i_example]['messages'][1:-1]]),
            actual_output=predicted_examples1[i_example]['messages'][-1]['content'],
            expected_output=gt_examples[i_example]['messages'][-1]['content'],
        )
        test_case2 = LLMTestCase(
            input="\n".join([msg['content'] for msg in predicted_examples2[i_example]['messages'][1:-1]]),
            actual_output=predicted_examples2[i_example]['messages'][-1]['content'],
            expected_output=gt_examples[i_example]['messages'][-1]['content'],
        )

        style_matching_metric_geval.measure(test_case1)
        predicted_examples1[i_example]['fun_score'] = style_matching_metric_geval.score
        scores1.append(style_matching_metric_geval.score)

        style_matching_metric_geval.measure(test_case2)
        predicted_examples2[i_example]['fun_score'] = style_matching_metric_geval.score
        scores2.append(style_matching_metric_geval.score)

        humor_quality_metric.measure(test_case1)
        predicted_examples1[i_example]['humor_quality'] = humor_quality_metric.score
        scores1_quality.append(humor_quality_metric.score)

        humor_quality_metric.measure(test_case2)
        predicted_examples2[i_example]['humor_quality'] = humor_quality_metric.score
        scores2_quality.append(humor_quality_metric.score)

    return gt_examples, predicted_examples1, predicted_examples2, scores1, scores2, scores1_quality, scores2_quality


def main(args):
    gt_examples = prepare_openai_jokes_dataset(args)
    print("============== GT EXAMPLES ===============")
    for i_example in range(len(gt_examples)):
        print(gt_examples[i_example])
        print("-----------")
    predicted_examples1 = get_model_predictions(gt_examples, system_message1)
    predicted_examples2 = get_model_predictions(gt_examples, system_message2)
    print("=============== GT and Predicted EXAMPLES ======================")
    for i_example in range(len(gt_examples)):
        print(gt_examples[i_example])
        print(predicted_examples1[i_example])
        print(predicted_examples2[i_example])
        print("-----------")
    gt_examples, predicted_examples1, predicted_examples2, scores1, scores2, scores1_quality, scores2_quality = (
        evaluate_metrics(gt_examples, predicted_examples1, predicted_examples2))

    print("=============== GT and Predicted EXAMPLES with METRICS ======================")
    for i_example in range(len(gt_examples)):
        print(gt_examples[i_example])
        print(predicted_examples1[i_example])
        print(predicted_examples2[i_example])
        print("-----------")

    print("=============== Scores ======================")
    ttest_res1 = ttest_1samp(scores1, 0.5)
    print("SCORE1:", np.mean(scores1), scores1)
    print("TTEST1:", ttest_res1)
    ttest_res2 = ttest_1samp(scores2, 0.5)
    print("SCORE2:", np.mean(scores2), scores2)
    print("TTEST2:", ttest_res2)

    print("*********=============== My metrics ======================*********")

    print("SCORE1 Humor Quality:", np.mean(scores1_quality), scores1_quality)
    print("TTEST1 Humor Quality:", ttest_1samp(scores1_quality, 0.5))
    print("SCORE2 Humor Quality:", np.mean(scores2_quality), scores2_quality)
    print("TTEST2 Humor Quality:", ttest_1samp(scores2_quality, 0.5))

    rel_ttest_res = ttest_rel(scores1, scores2)

    print("=============== Rel Scores ======================")
    print("TTEST style:", rel_ttest_res)
    print("TTEST Humor Quality:", ttest_rel(scores1_quality, scores2_quality))
    print("*********=============== My metrics ======================*********")


if __name__ == '__main__':
    class Args:
        num_samples = 20
    args = Args()
    main(args)

============== GT EXAMPLES ===============
{'messages': [{'role': 'system', 'content': 'Ты чат бот, который генерирует анекдоты. Продолжи диалог одной репликой так, чтоб получился анекдот. Example input: - Я не могу выйти на работу, потому что жена сломала два ребра. - А при чём тут ты? Expected output: - Мои!'}, {'role': 'assistant', 'content': 'Ювелирный магазин. Менеджер продавцу:'}, {'role': 'user', 'content': 'Как вам удается продавать столько бриллиантов?'}, {'role': 'assistant', 'content': 'Я просто говорю мужчинам, что жену в бриллиантах нельзя одну выпускать на улицу - убьют.', 'weight': 1}]}
-----------
{'messages': [{'role': 'system', 'content': 'Ты чат бот, который генерирует анекдоты. Продолжи диалог одной репликой так, чтоб получился анекдот. Example input: - Я не могу выйти на работу, потому что жена сломала два ребра. - А при чём тут ты? Expected output: - Мои!'}, {'role': 'user', 'content': 'Одесса. Маленький дворик. На балкон второго этажа выходит\nСара в халате, пове

Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

=============== GT and Predicted EXAMPLES ======================
{'messages': [{'role': 'system', 'content': 'Ты чат бот, который генерирует анекдоты. Продолжи диалог одной репликой так, чтоб получился анекдот. Example input: - Я не могу выйти на работу, потому что жена сломала два ребра. - А при чём тут ты? Expected output: - Мои!'}, {'role': 'assistant', 'content': 'Ювелирный магазин. Менеджер продавцу:'}, {'role': 'user', 'content': 'Как вам удается продавать столько бриллиантов?'}, {'role': 'assistant', 'content': 'Я просто говорю мужчинам, что жену в бриллиантах нельзя одну выпускать на улицу - убьют.', 'weight': 1}]}
{'messages': [{'role': 'system', 'content': 'Ты чат бот, который генерирует анекдоты. Продолжи диалог одной репликой так, чтоб получился анекдот. Example input: - Я не могу выйти на работу, потому что жена сломала два ребра. - А при чём тут ты? Expected output: - Мои!'}, {'role': 'assistant', 'content': 'Ювелирный магазин. Менеджер продавцу:'}, {'role': 'user', 'cont

Output()

...

Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

=============== GT and Predicted EXAMPLES with METRICS ======================
{'messages': [{'role': 'system', 'content': 'Ты чат бот, который генерирует анекдоты. Продолжи диалог одной репликой так, чтоб получился анекдот. Example input: - Я не могу выйти на работу, потому что жена сломала два ребра. - А при чём тут ты? Expected output: - Мои!'}, {'role': 'assistant', 'content': 'Ювелирный магазин. Менеджер продавцу:'}, {'role': 'user', 'content': 'Как вам удается продавать столько бриллиантов?'}, {'role': 'assistant', 'content': 'Я просто говорю мужчинам, что жену в бриллиантах нельзя одну выпускать на улицу - убьют.', 'weight': 1}]}
{'messages': [{'role': 'system', 'content': 'Ты чат бот, который генерирует анекдоты. Продолжи диалог одной репликой так, чтоб получился анекдот. Example input: - Я не могу выйти на работу, потому что жена сломала два ребра. - А при чём тут ты? Expected output: - Мои!'}, {'role': 'assistant', 'content': 'Ювелирный магазин. Менеджер продавцу:'}, {'role': 

Стиль:
Промт 1 имеет среднее значение 0.481, что немного выше порогового значения 0.5, с p-value 0.5839, что указывает на отсутствие значительных отличий. Промт 2, с результатом 0.293, значительно уступает, с p-value 1.91e-09, что подтверждает сильное отличие от 0.5. Это означает, что Промт 1 лучше соответствует критерию стиля.

Качество юмора:
Промт 1 также превосходит Промт 2 по качеству юмора, с результатом 0.451 (p-value = 0.082), что близко к значению 0.5, но не является статистически значимым. Промт 2 с результатом 0.298 и p-value 5.08e-07 сильно отличается от порогового значения, что указывает на его слабое соответствие критерию юмора.

Смею предположить, что промт 1 справляется со своей задачей лучше чем, промт 2

# Дообучение модели:

In [15]:
def convert_to_chat_format_random(df, n=50):
    random_rows = df.sample(n=n, random_state=42)
    
    chat_format = []
    for _, row in random_rows.iterrows():
        chat_format.append({
            "messages": [
                {"role": "system", "content": "You are a humorous assistant, make me laugh."},
                {"role": "user", "content": row['context']},
                {"role": "assistant", "content": row['utterance']}
            ]
        })
    
    return chat_format

In [16]:
dataset = load_dataset("inkoziev/jokes_dialogues")

df = dataset['train'].to_pandas()
chat_data = convert_to_chat_format_random(df)

with open("jokes_for_finetuning_chat.jsonl", "w", encoding="utf-8") as f:
    for item in chat_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

In [17]:
from openai import OpenAI

client = OpenAI()
client.files.create(file=open("jokes_for_finetuning_chat.jsonl", "rb"), purpose="fine-tune")

FileObject(id='file-SKMrMKaFrxt4sjPk5G6q6p', bytes=27585, created_at=1734984771, filename='jokes_for_finetuning_chat.jsonl', object='file', purpose='fine-tune', status='processed', status_details=None)

In [18]:
client.fine_tuning.jobs.create(
    training_file="file-SKMrMKaFrxt4sjPk5G6q6p", model="gpt-4o-mini-2024-07-18"
)

FineTuningJob(id='ftjob-sFhPtCMP6URZQ4LZFISv4HnP', created_at=1734984832, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-yyyPGCw9VCJjgaycEyhn8xFf', result_files=[], seed=1610405273, status='validating_files', trained_tokens=None, training_file='file-SKMrMKaFrxt4sjPk5G6q6p', validation_file=None, estimated_finish=None, integrations=[], method=Method(dpo=None, supervised=MethodSupervised(hyperparameters=MethodSupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto')), type='supervised'), user_provided_suffix=None)

In [24]:
print(client.fine_tuning.jobs.retrieve("ftjob-sFhPtCMP6URZQ4LZFISv4HnP").status)
print(client.fine_tuning.jobs.retrieve("ftjob-sFhPtCMP6URZQ4LZFISv4HnP").fine_tuned_model)

succeeded
ft:gpt-4o-mini-2024-07-18:personal::AhjBubwt


# Прогон дообученной модели:

In [25]:
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

client = OpenAI()

style_matching_metric_geval = GEval(
    name="Style Matching",
    criteria="Определи, насколько последняя фраза (actual output) делает анекдот смешным, логичным и остроумным:",

    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT]
)

humor_quality_metric = GEval(
    name="Humor Quality",
    criteria="Оцени, насколько последняя фраза (actual output) смешная. "
             "Учитывай качество шутки, её оригинальность, уместность и способность вызвать улыбку. "
             "Ответ должен быть логичным и доставлять удовольствие при чтении.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT]
)

system_message1 = {"role": "system",
                   "content": "Ты чат бот, который генерирует анекдоты. "
                              "Продолжи диалог одной репликой так, чтоб получился анекдот. "
                              "Example input: - Я не могу выйти на работу, потому что жена сломала два ребра. - А при чём тут ты? "
                              "Expected output: - Мои!"}
system_message2 = {"role": "system",
                   "content": "Ты чат бот, который генерирует диалоги. Продолжи диалог"}


def prepare_openai_jokes_dataset(args):
    ds = load_dataset("inkoziev/jokes_dialogues")
    ds_df = ds['train'].to_pandas()

    # print(ds_df)
    unique_chat_ids = ds_df['src_hash'].unique()
    # unique_chat_ids = set(ds['train']['src_hash'])
    sampled_chat_ids = np.random.choice(list(unique_chat_ids), args.num_samples, replace=False)

    openai_chat_examples = []
    for chat_id in sampled_chat_ids:
        reply_nums = ds_df[ds_df['src_hash'] == chat_id]['reply_num'].values
        reply_nums = np.sort(reply_nums)
        messages = [system_message1]
        if reply_nums[0] != 1:
            continue
        message1 = ds_df[(ds_df['src_hash'] == chat_id) & (ds_df['reply_num'] == 1)]['context'].values[0]
        messages.append({"role": "assistant" if 0 % 2 == reply_nums[-1] % 2 else "user",
                         "content": message1})

        for reply_num in reply_nums:
            message = ds_df[(ds_df['src_hash'] == chat_id) & (ds_df['reply_num'] == reply_num)]['utterance'].values[0]
            message_metadata = {"role": "assistant" if reply_num % 2 == reply_nums[-1] % 2 else "user",
                                "content": message}
            if message_metadata['role'] == 'assistant':
                if reply_num == reply_nums[-1]:
                    message_metadata['weight'] = 1
                else:
                    message_metadata['weight'] = 0
            messages.append(message_metadata)

        openai_chat_example = {"messages": messages}
        openai_chat_examples.append(openai_chat_example)
    return openai_chat_examples


def get_model_predictions(gt_examples, system_message):
    predicted_examples = []
    for i_example in range(len(gt_examples)):
        completion = client.chat.completions.create(
            model="ft:gpt-4o-mini-2024-07-18:personal::AhjBubwt", # Это моя моделька
            messages=[system_message] + gt_examples[i_example]['messages'][1:-1],
        )
        # predicted_examples.append(completion.choices[0].message.content)
        predicted_example = [system_message] + gt_examples[i_example]['messages'][1:-1]
        predicted_example.append({"role": "assistant", "content": completion.choices[0].message.content})
        predicted_example = {"messages": predicted_example}
        predicted_examples.append(predicted_example)

    return predicted_examples


def evaluate_metrics(gt_examples, predicted_examples1, predicted_examples2):
    scores1 = []
    scores2 = []
    scores1_quality = []
    scores2_quality = []

    for i_example in range(len(gt_examples)):
        test_case1 = LLMTestCase(
            input="\n".join([msg['content'] for msg in predicted_examples1[i_example]['messages'][1:-1]]),
            actual_output=predicted_examples1[i_example]['messages'][-1]['content'],
            expected_output=gt_examples[i_example]['messages'][-1]['content'],
        )
        test_case2 = LLMTestCase(
            input="\n".join([msg['content'] for msg in predicted_examples2[i_example]['messages'][1:-1]]),
            actual_output=predicted_examples2[i_example]['messages'][-1]['content'],
            expected_output=gt_examples[i_example]['messages'][-1]['content'],
        )

        style_matching_metric_geval.measure(test_case1)
        predicted_examples1[i_example]['fun_score'] = style_matching_metric_geval.score
        scores1.append(style_matching_metric_geval.score)

        style_matching_metric_geval.measure(test_case2)
        predicted_examples2[i_example]['fun_score'] = style_matching_metric_geval.score
        scores2.append(style_matching_metric_geval.score)

        humor_quality_metric.measure(test_case1)
        predicted_examples1[i_example]['humor_quality'] = humor_quality_metric.score
        scores1_quality.append(humor_quality_metric.score)

        humor_quality_metric.measure(test_case2)
        predicted_examples2[i_example]['humor_quality'] = humor_quality_metric.score
        scores2_quality.append(humor_quality_metric.score)

    return gt_examples, predicted_examples1, predicted_examples2, scores1, scores2, scores1_quality, scores2_quality


def main(args):
    gt_examples = prepare_openai_jokes_dataset(args)
    print("============== GT EXAMPLES ===============")
    for i_example in range(len(gt_examples)):
        print(gt_examples[i_example])
        print("-----------")
    predicted_examples1 = get_model_predictions(gt_examples, system_message1)
    predicted_examples2 = get_model_predictions(gt_examples, system_message2)
    print("=============== GT and Predicted EXAMPLES ======================")
    for i_example in range(len(gt_examples)):
        print(gt_examples[i_example])
        print(predicted_examples1[i_example])
        print(predicted_examples2[i_example])
        print("-----------")
    gt_examples, predicted_examples1, predicted_examples2, scores1, scores2, scores1_quality, scores2_quality = (
        evaluate_metrics(gt_examples, predicted_examples1, predicted_examples2))

    print("=============== GT and Predicted EXAMPLES with METRICS ======================")
    for i_example in range(len(gt_examples)):
        print(gt_examples[i_example])
        print(predicted_examples1[i_example])
        print(predicted_examples2[i_example])
        print("-----------")

    print("=============== Scores ======================")
    ttest_res1 = ttest_1samp(scores1, 0.5)
    print("SCORE1:", np.mean(scores1), scores1)
    print("TTEST1:", ttest_res1)
    ttest_res2 = ttest_1samp(scores2, 0.5)
    print("SCORE2:", np.mean(scores2), scores2)
    print("TTEST2:", ttest_res2)

    print("*********=============== My metrics ======================*********")

    print("SCORE1 Humor Quality:", np.mean(scores1_quality), scores1_quality)
    print("TTEST1 Humor Quality:", ttest_1samp(scores1_quality, 0.5))
    print("SCORE2 Humor Quality:", np.mean(scores2_quality), scores2_quality)
    print("TTEST2 Humor Quality:", ttest_1samp(scores2_quality, 0.5))

    rel_ttest_res = ttest_rel(scores1, scores2)

    print("=============== Rel Scores ======================")
    print("TTEST style:", rel_ttest_res)
    print("TTEST Humor Quality:", ttest_rel(scores1_quality, scores2_quality))
    print("*********=============== My metrics ======================*********")


if __name__ == '__main__':
    class Args:
        num_samples = 20
    args = Args()
    main(args)

============== GT EXAMPLES ===============
{'messages': [{'role': 'system', 'content': 'Ты чат бот, который генерирует анекдоты. Продолжи диалог одной репликой так, чтоб получился анекдот. Example input: - Я не могу выйти на работу, потому что жена сломала два ребра. - А при чём тут ты? Expected output: - Мои!'}, {'role': 'user', 'content': 'Одного известного либерального экономиста пригласили прочесть лекцию о глобализации мировой экономики перед рабочей аудиторией.'}, {'role': 'assistant', 'content': 'Как же вам проще объяснить сущность этого процесса? Вот представьте себе, что в нашей стране производятся глиняные свистки, которые поставляются в Монголию для местных чабанов. Чабаны разводят баранов, шкуры которых поставляются...', 'weight': 0}, {'role': 'user', 'content': 'Нам?'}, {'role': 'assistant', 'content': 'Нет. В том-то и смысл глобализации, что монголы поставляют эти шкуры в Грецию, где их соответствующим образом выделывают, получают превосходный мех и поставляют его...', 'w

Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

=============== GT and Predicted EXAMPLES ======================
{'messages': [{'role': 'system', 'content': 'Ты чат бот, который генерирует анекдоты. Продолжи диалог одной репликой так, чтоб получился анекдот. Example input: - Я не могу выйти на работу, потому что жена сломала два ребра. - А при чём тут ты? Expected output: - Мои!'}, {'role': 'user', 'content': 'Одного известного либерального экономиста пригласили прочесть лекцию о глобализации мировой экономики перед рабочей аудиторией.'}, {'role': 'assistant', 'content': 'Как же вам проще объяснить сущность этого процесса? Вот представьте себе, что в нашей стране производятся глиняные свистки, которые поставляются в Монголию для местных чабанов. Чабаны разводят баранов, шкуры которых поставляются...', 'weight': 0}, {'role': 'user', 'content': 'Нам?'}, {'role': 'assistant', 'content': 'Нет. В том-то и смысл глобализации, что монголы поставляют эти шкуры в Грецию, где их соответствующим образом выделывают, получают превосходный мех и 

Output()

...

Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

=============== GT and Predicted EXAMPLES with METRICS ======================
{'messages': [{'role': 'system', 'content': 'Ты чат бот, который генерирует анекдоты. Продолжи диалог одной репликой так, чтоб получился анекдот. Example input: - Я не могу выйти на работу, потому что жена сломала два ребра. - А при чём тут ты? Expected output: - Мои!'}, {'role': 'user', 'content': 'Одного известного либерального экономиста пригласили прочесть лекцию о глобализации мировой экономики перед рабочей аудиторией.'}, {'role': 'assistant', 'content': 'Как же вам проще объяснить сущность этого процесса? Вот представьте себе, что в нашей стране производятся глиняные свистки, которые поставляются в Монголию для местных чабанов. Чабаны разводят баранов, шкуры которых поставляются...', 'weight': 0}, {'role': 'user', 'content': 'Нам?'}, {'role': 'assistant', 'content': 'Нет. В том-то и смысл глобализации, что монголы поставляют эти шкуры в Грецию, где их соответствующим образом выделывают, получают превос

Стиль:
SCORE1: 0.321, p-value: 1.94e-06
SCORE2: 0.340, p-value: 4.85e-05
TTEST style: p-value: 0.49
Качество юмора:
SCORE1 Humor Quality: 0.322, p-value: 3.24e-06
SCORE2 Humor Quality: 0.339, p-value: 5.93e-05
TTEST Humor Quality: p-value: 0.64

В новых результатах различия между двумя промтами стали менее выраженными по сравнению с предыдущими тестами.

Ранее Промт 1 заметно выигрывал в обеих категориях — и по стилю, и по качеству юмора, с результатами, значительно отличающимися от порогового значения. Однако в новых метках мы видим, что разница между Промтом 1 и Промтом 2 уменьшилась. Хотя Промт 1 все еще показывает немного лучшие результаты по стилю и качеству юмора, p-value для обоих тестов теперь выше, что указывает на менее значительные различия. В целом, по сравнению с предыдущими метками, улучшений в явном виде нет, но и разрыв между промтами значительно сократился.

Будто-то бы, дообучение сделало только хуже...